In [1]:
# Learning Path Recommendation System
# Collaborative Filtering (Matrix Factorization)

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------
# Sample intern-course ratings data
# (rows = interns, columns = courses/modules, values = ratings/engagement)
# ----------------------------
data = {
    "Python Basics":     [5, 0, 4, 0, 3],
    "Data Science":      [4, 0, 0, 2, 0],
    "Machine Learning":  [0, 3, 5, 0, 4],
    "Deep Learning":     [0, 4, 0, 0, 5],
    "SQL":               [3, 0, 4, 0, 0]
}

interns = ["Intern_1", "Intern_2", "Intern_3", "Intern_4", "Intern_5"]
ratings_df = pd.DataFrame(data, index=interns)

print("=== Intern-Course Ratings Matrix ===")
print(ratings_df)

# ----------------------------
# Step 1: Matrix Factorization (Collaborative Filtering)
# ----------------------------
from sklearn.decomposition import NMF

# Apply Non-negative Matrix Factorization
nmf = NMF(n_components=2, random_state=42)
user_features = nmf.fit_transform(ratings_df.values)  # intern latent features
course_features = nmf.components_                    # course latent features

# Reconstructed rating predictions
predicted_ratings = np.dot(user_features, course_features)

predicted_df = pd.DataFrame(predicted_ratings, index=interns, columns=ratings_df.columns)
print("\n=== Predicted Ratings Matrix ===")
print(predicted_df.round(2))

# ----------------------------
# Step 2: Recommend top courses for each intern
# ----------------------------
def recommend_courses(intern, num_recommendations=2):
    # Get actual and predicted ratings
    actual_ratings = ratings_df.loc[intern]
    predicted_ratings = predicted_df.loc[intern]

    # Exclude already taken courses
    unrated = actual_ratings[actual_ratings == 0].index
    recommendations = predicted_ratings[unrated].sort_values(ascending=False)

    return recommendations.head(num_recommendations)

# Example: Get recommendations for Intern_2
print("\n=== Recommendations ===")
for intern in interns:
    print(f"{intern} → {recommend_courses(intern, 2).index.tolist()}")

=== Intern-Course Ratings Matrix ===
          Python Basics  Data Science  Machine Learning  Deep Learning  SQL
Intern_1              5             4                 0              0    3
Intern_2              0             0                 3              4    0
Intern_3              4             0                 5              0    4
Intern_4              0             2                 0              0    0
Intern_5              3             0                 4              5    0

=== Predicted Ratings Matrix ===
          Python Basics  Data Science  Machine Learning  Deep Learning   SQL
Intern_1           4.77          2.47              1.56           0.00  3.67
Intern_2           0.98          0.00              3.24           3.50  0.00
Intern_3           4.40          2.02              2.94           1.80  3.00
Intern_4           0.53          0.27              0.17           0.00  0.41
Intern_5           2.11          0.41              4.61           4.69  0.61

=== Recomm